# धडा 12 - एजंट स्क्रॅचपॅडसह चॅट इतिहास कमी करणे

हा नोटबुक Microsoft Agent Framework वापरून लांबच्या संभाषणांमध्ये संदर्भ कसा व्यवस्थापित करायचा हे दर्शवितो. संभाषणे वाढत गेल्यामुळे टोकन गणना वाढते — शेवटी मॉडेलच्या संदर्भ विंडोपेक्षा जास्त होते. याला आपण **संदर्भ सारांश देण्याच्या पद्धतीने** आणि **एजंट स्क्रॅचपॅड** वापरून निरंतर आठवण ठेवून हाताळतो.

## तुम्हाला काय शिकायला मिळेल:
1. **का संदर्भ व्यवस्थापन महत्त्वाचे आहे**: टोकन मर्यादा आणि संदर्भ विंडोज समजणे
2. **संदर्भ-जाणकार एजंट**: असे एजंट तयार करणे जे स्वतःच्या संभाषणाचा संदर्भ हाताळतात
3. **संदर्भ सारांश देण्याची पद्धत**: संभाषण इतिहास संक्षिप्त करण्यासाठी साधने वापरणे
4. **एजंट स्क्रॅचपॅड**: संदर्भ कमी झाल्यानंतरही टिकणारी आठवण

## पूर्वशर्त:
- Azure OpenAI सेटअप आणि पर्यावरण चल निश्चित केलेले 
- मागील धड्यांमधून मूलभूत एजंट संकल्पनांची समज


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## का संदर्भ व्यवस्थापन महत्त्वाचे आहे

प्रत्येक LLM कडे एक मर्यादित **संदर्भ विंडो** असते — एका विनंतीत ती प्रक्रिया करू शकणाऱ्या टोकनची कमाल संख्या. एक मल्टि-टर्न संवाद जसजसा पुढे सरकतो:

- **टोकन संख्या रेषीय रूपात वाढते** प्रत्येक वापरकर्त्याच्या संदेश आणि सहाय्यकाच्या प्रतिसादासह.
- **प्रॉम्प्ट टोकनची किंमत जास्त असते** कारण संपूर्ण इतिहास प्रत्येक टर्नला परत पाठवला जातो.
- शेवटी संभाषण **संदर्भ विंडो ओलांडते** आणि मॉडेल कापणी करते किंवा त्रुटी देते.

### संदर्भ व्यवस्थापनासाठी धोरणे

| धोरण | कसे कार्य करते | समतोल |
|---|---|---|
| **कापणी** | सर्वात जुनी संदेशे वगळा | सुरुवातीचा संदर्भ गमावतो |
| **सारांश** | जुन्या संदेशांचा सारांश तयार करा | काही तपशील हरवतो, पण मुख्य मुद्दे ठेवल्या जातात |
| **स्क्रॅचपॅड / बाह्य मेमरी** | संभाषणाबाहेर मुख्य तथ्ये साठवा | उपकरण कॉलची गरज असते, पण कोणत्याही कमी होण्यावर टिकून राहतो |

या नोटबुकमध्ये आम्ही **सारांश** आणि **स्क्रॅचपॅड टूल** यांचा संयोग करतो जेणेकरून एजंट संभाषणाचा इतिहास संक्षिप्त झाल्यावरही सलगता राखू शकेल.


## संदर्भ-जाणणारा एजंट तयार करणे


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## एका लांबच्या संभाषणाची अनुकरण करणे

चला एक बहु-चरणीय संभाषण पाहूया ज्यामुळे कसे संदर्भ जमा होतात ते समजेल. एजंटने महत्त्वाच्या तपशीलांना (प्राधान्ये, बजेट, प्रवासाच्या तारखा) प्रत्येक टप्प्यावर जपावे आणि सलगपणाचे प्रदर्शन करावे.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

एजंट कसा मागील संवादातील संदर्भ जपतो याकडे लक्ष द्या — त्याला जपान, सुशी, मंदिरे, छायाचित्रण, $3000 बजेट, एकटे प्रवास, आणि एप्रिलचा वेळ असा सर्व काही माहित आहे. एका लहान संभाषणात हे चांगले कार्य करते, परंतु संवाद वाढत गेल्यानंतर पूर्ण इतिहास पुन्हा पाठवणे महागडे होते.

चला, अधिक संवाद घेऊन संदर्भ संचित होतो कसा ते पाहूया:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## संदर्भ संक्षेपण नमुना

संवाद जसा वाढतो, तसा आपण संकलित संदर्भ एक संक्षिप्त स्वरूपात संक्षेपित करण्यासाठी **संक्षेपण साधनाचा** वापर करू शकतो. एजंट हा साधन कॉल करतो जेणेकरून आवश्यक प्राधान्यांची नोंद करता येईल, त्यामुळे जुने संदेश नष्ट झाले तरीही महत्वाची माहिती राखली जाईल.

हा नमुना अधिक प्रगत इतिहास कमी करण्यासाठीचा बांधकामाचा घटक आहे:
1. एजंट संवादातून मुख्य तथ्ये ओळखतो
2. त्याने ती टिकवून ठेवण्यासाठी संक्षेपण साधन कॉल करते
3. जुने संदेश सुरक्षितपणे काढून टाकता येतात कारण संक्षेप नागाल महत्वाचे मुद्दे टिपतो

खाली आम्ही `summarize_preferences` साधन परिभाषित करतो ज्याला एजंट त्याने शिकलो त्या गोष्टींचा संक्षिप्त सारांश नोंदवण्यासाठी कॉल करू शकतो.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## सारांश

या धड्यात आपण Microsoft Agent Framework वापरून दीर्घकालीन एजंट संवादांमध्ये संदर्भ कसा व्यवस्थापित करायचा हे शिकलात:

### मुख्य संकल्पना
- **संदर्भ विंडो मर्यादित असतात** — संभाषण इतिहासातील प्रत्येक टोकनचे पैसे लागतात आणि ते मर्यादेत गणले जातात.
- **सारांशकरण साधने** एजंटला जमा केलेला संदर्भ संक्षिप्त सारांशांमध्ये रूपांतरित करण्याची परवानगी देतात, ज्यामुळे टोकन वापर कमी होतो आणि आवश्यक माहिती टिकवून राहते.
- **एजंट स्क्रॅचपॅड्स** हे कायमस्वरूपी बाह्य स्मृती प्रदान करतात जे कोणत्याही संभाषण संक्षेपणानंतर कायम राहते.

### आपण काय तयार केले
- एक **संदर्भ-जाणकार एजंट** जो बहु-टर्न संवादांमध्ये सलगता राखतो
- एक **सारांशकरण साधन** (`summarize_preferences`) जे महत्त्वाच्या वापरकर्ता तपशीलांचे संक्षिप्त रेकॉर्ड ठेवते
- संदर्भ जपणे आणि बदल हाताळणे यांचे प्रदर्शन करणारा **बहु-टर्न संवाद**

### प्रत्यक्ष अर्ज
- **ग्राहक सेवा बॉट्स**: लांब समर्थन सत्रांमध्ये पसंती लक्षात ठेवतात
- **वैयक्तिक सहाय्यक**: संदर्भ पुन्हा समजावून न देता चालू प्रकल्प ट्रॅक करतात
- **शैक्षणिक शिक्षक**: अनेक संवादांमध्ये विद्यार्थी प्रगती कायम ठेवतात

### पुढील पावले
- फाईल-आधारित कायमस्वरूपीसह संपूर्ण स्क्रॅचपॅड साधन अंमलात आणा
- सारांशकरणानंतर स्वयंचलित इतिहास संक्षेपणा जोडा
- अर्थपूर्ण स्मृती शोधासाठी व्हेक्टर डेटाबेससह संयोजन करा
- पूर्ण संदर्भासह दिवसांनंतर संवाद पुन्हा सुरू करू शकणारे एजंट तयार करा


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित करण्यात आला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात ठेवा की स्वयंचलित अनुवादांमध्ये चुका किंवा अचूकतेच्या त्रुटी असू शकतात. मूळ दस्तऐवज त्याच्या स्थानिक भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाच्या माहिती साठी व्यावसायिक मानवी अनुवादाची शिफारस केली जाते. या अनुवादाच्या वापरामुळे उद्भवलेल्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थव्यक्तीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
